In [1]:
import os
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MySPARK") \
    .config("spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
        "ru.yandex.clickhouse:clickhouse-jdbc:0.3.2,"
        "org.postgresql:postgresql:42.5.0,"
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0",
        ) \
    .getOrCreate()


hadoop_conf = spark._jsc.hadoopConfiguration()
hadoop_conf.set("fs.s3a.access.key", os.getenv("MINIO_ROOT_USER"))
hadoop_conf.set("fs.s3a.secret.key", os.getenv("MINIO_ROOT_PASSWORD"))
hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")
hadoop_conf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
hadoop_conf.set("fs.s3a.path.style.access", "true")

:: loading settings :: url = jar:file:/usr/local/lib/python3.11/dist-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
ru.yandex.clickhouse#clickhouse-jdbc added as a dependency
org.postgresql#postgresql added as a dependency
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-38df6f58-3912-4d74-80fd-2bc203ae5b48;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found ru.yandex.clickhouse#clickhouse-jdbc;0.3.2 in central
	found com.clickhouse#clickhouse-http-client;0.3.2 in central
	found com.clickhouse#clickhouse-client;0.3.2 in central
	found org.lz4#lz4-java;1.8.0 in central
	found com.google.code.gson#gson;2.8.8 in central
	found org.apache.httpcomponents#httpclient;4.5

In [2]:
jdbc_url = "jdbc:postgresql://postgres_source:5432/source" 
table_name = "public.shops" 
db_user = os.getenv("POSTGRES_USER") 
db_password = os.getenv("POSTGRES_PASSWORD") 
shops_df = spark.read \
                    .format("jdbc") \
                    .option("url", jdbc_url) \
                    .option("user", db_user) \
                    .option("password", db_password) \
                    .option("dbtable", table_name) \
                    .option("fetchsize", 1000) \
                    .option("driver", "org.postgresql.Driver") \
                    .load() 


In [3]:
jdbc_url = "jdbc:postgresql://postgres_source:5432/source" 
table_name = "public.shop_timezone" 
db_user = os.getenv("POSTGRES_USER") 
db_password = os.getenv("POSTGRES_PASSWORD") 
shop_timezone_df = spark.read \
				.format("jdbc") \
				.option("url", jdbc_url) \
				.option("user", db_user) \
				.option("password", db_password) \
				.option("dbtable", table_name) \
				.option("fetchsize", 1000) \
				.option("driver", "org.postgresql.Driver") \
				.load() 


In [16]:
shops_df.createOrReplaceTempView("shops") 
shop_timezone_df.createOrReplaceTempView("shop_timezone")

In [11]:
spark.sql("""
    WITH tz AS (
        SELECT
            CASE
                WHEN plant RLIKE '^[0-9]+$' THEN CAST(plant AS INT)
                ELSE NULL
            END AS plant_int,
            CASE
                WHEN time_zone RLIKE '^RUS[0-9]+$'
                    THEN CAST(substring(time_zone, 4) AS INT)
                ELSE NULL
            END AS tz_num
        FROM shop_timezone
    )

    SELECT 
        ps.st_id,
        ps.shop_name,
        CASE
            WHEN tz.tz_num IS NULL THEN 3
            ELSE tz.tz_num
        END AS tz_code
    FROM shops ps
    INNER JOIN tz 
        ON ps.st_id = tz.plant_int
    ORDER BY ps.st_id
    LIMIT 100
""").show()



+-----+-----------+-------+
|st_id|  shop_name|tz_code|
+-----+-----------+-------+
|  842|      Lenta|      3|
|  842|      Lenta|      7|
|  843|     Magnit|      4|
|  844|       Spar|      3|
|  845|Pyaterochka|      3|
|  845|Pyaterochka|      5|
|  847|      Diksi|      3|
|  848|      Lenta|      8|
|  848|      Lenta|      3|
|  848|      Lenta|      3|
+-----+-----------+-------+



In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

tz_df = (
    shop_timezone_df
    .withColumn(
        "plant_int",
        F.when(F.col("plant").rlike("^[0-9]+$"), F.col("plant").cast(IntegerType()))
         .otherwise(None)
    )
    .withColumn(
        "tz_num",
        F.when(F.col("time_zone").rlike("^RUS[0-9]+$"), 
               F.substring(F.col("time_zone"), 4, 2).cast(IntegerType()))
         .otherwise(None)
    )
    .select("plant_int", "tz_num")
)

result_df = (
    shops_df.alias("ps")
    .join(tz_df.alias("tz"), F.col("ps.st_id") == F.col("tz.plant_int"), "inner")
    .withColumn(
        "tz_code",
        F.coalesce(F.col("tz.tz_num"), F.lit(3))
    )
    .select("ps.st_id", "ps.shop_name", "tz_code")
    .orderBy("ps.st_id")
    .limit(100)
)

result_df.show()



+-----+-----------+-------+
|st_id|  shop_name|tz_code|
+-----+-----------+-------+
|  842|      Lenta|      3|
|  842|      Lenta|      7|
|  843|     Magnit|      4|
|  844|       Spar|      3|
|  845|Pyaterochka|      3|
|  845|Pyaterochka|      5|
|  847|      Diksi|      3|
|  848|      Lenta|      8|
|  848|      Lenta|      3|
|  848|      Lenta|      3|
+-----+-----------+-------+



In [ ]:
spark.stop()